<a href="https://colab.research.google.com/github/cfbarrerab/AND/blob/main/Generador_de_Gr%C3%A1ficos_M1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [122]:

# ==========================================
# 0. CONFIGURACIÓN LIBRERIAS Y FUENTES
# ==========================================
import os
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Definir la jerarquía tipográfica con respaldo automático
# Si Aptos no está instalada, pasará automáticamente a Arial/DejaVu Sans sin fallar
plt.rcParams["font.sans-serif"] = [
    "Aptos",
    "Aptos Display",
    "Arial",
    "DejaVu Sans",
]
plt.rcParams["font.family"] = "sans-serif"

In [129]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.cm as cm

# ==========================================
# 1. CONFIGURACIÓN Y PALETA "NEGRO ORO" HEX
# ==========================================
OUTPUT_DIR = "graficos_negro_oro"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Paleta 1: Ejecutivo Azul y Oro
COLOR_CAPA_5 = "#1B2A4A"  # Azul marino muy oscuro
COLOR_CAPA_4 = "#2C3E6B"
COLOR_CAPA_3 = "#415A8D"
COLOR_CAPA_2 = "#657FB2"
COLOR_CAPA_1 = "#A1B4D5"  # Azul claro centro
COLOR_PUNTAJE = "#D4AF37" # Dorado brillante para la línea

# Color de los Datos (Puntaje)
COLOR_PUNTAJE = "#D4AF37"  # Oro Brillante (Metallic Gold)
COLOR_RELLENO_PUNTAJE = "#D4AF37" # Mismo Oro para relleno con transparencia

# Textos sobre fondo oscuro
COLOR_TEXTO_GRAFICO = "#FFFFFF" # Blanco

# --- Paleta Gráfica de Barras ---
COLOR_FONDO_BARRAS = "#1A1A1A" # Gris casi negro para el fondo del ax
COLOR_BARRA = "#C5A028"       # Oro Satinado (ligeramente más mate)
COLOR_TEXTO_BARRAS = "#FFFFFF"  # Blanco


# ==========================================
# 2. MAPEO DE SUBDIMENSIONES Y PREGUNTAS
# ==========================================
# (Se mantiene igual, ajusta según tu CSV)
SUBDIMENSIONES = {
    "Subdimensión A: Control y Autonomía": [
        "Restricción de Libertad",
        "Sensibilidad al Rechazo y Poder",
    ],
    "Subdimensión B: Cumplimiento y Normas": [
        "Obediencia y Cumplimiento",
        "Actitud ante la Autoridad y el CRM",
    ],
    "Subdimensión C: Relaciones y Manejo de Conflicto": [
        "Confrontación y Refutación",
    ],
}


# ==========================================
# 3. FUNCIÓN 1: GRÁFICO POLAR (Negro Oro)
# ==========================================
def crear_grafico_polar_subdimensiones(sujeto_id, puntajes_subdim):
    categorías = list(puntajes_subdim.keys())
    valores = list(puntajes_subdim.values())
    N = len(categorías)

    # Reordenar/cerrar el círculo
    angulos = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angulos += angulos[:1]
    valores_cerrados = valores + valores[:1]

    # Crear figura con fondo oscuro
    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True), facecolor=COLOR_CAPA_5)
    ax.set_facecolor(COLOR_CAPA_5) # Fondo interno

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 30)
    ax.grid(False)
    ax.spines["polar"].set_visible(False)

    # Capas de fondo concéntricas (Degradado Negro)
    capas = [
        (30, COLOR_CAPA_5),
        (25, COLOR_CAPA_4),
        (20, COLOR_CAPA_3),
        (15, COLOR_CAPA_2),
        (10, COLOR_CAPA_1),
    ]
    for radio, color in capas:
        r = [radio] * (N + 1)
        ax.fill(angulos, r, color=color, zorder=1)

    # Polígono de datos del sujeto (Oro)
    ax.plot(
        angulos,
        valores_cerrados,
        color=COLOR_PUNTAJE,
        linewidth=4, # Un poco más grueso para destacar
        zorder=3,
        marker="o",
        markersize=8,
        label="Puntaje General",
    )
    # Relleno del puntaje con transparencia oro
    ax.fill(angulos, valores_cerrados, color=COLOR_relleno_PUNTAJE, alpha=0.3, zorder=2)

    # Etiquetas de las 3 subdimensiones (Texto Blanco)
    labels_format = [cat.replace(": ", ":\n") for cat in categorías]
    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(
        labels_format, fontsize=10, color=COLOR_TEXTO_GRAFICO, fontweight="bold"
    )

    # Escala radial (Texto Blanco)
    ax.set_yticks([0, 5, 10, 15, 20, 25, 30])
    ax.set_yticklabels(
        ["0", "5", "10", "15", "20", "25", "30"], color="#AAAAAA", fontsize=8 # Gris claro para la escala
    )
    ax.set_rlabel_position(0)

    # Título (Texto Blanco)
    plt.title(
        f"Perfil General por Subdimensiones - {sujeto_id}",
        fontsize=14,
        fontweight="bold",
        pad=30,
        color=COLOR_TEXTO_GRAFICO,
    )

    # Leyenda (Texto Blanco, fondo oscuro)
    leg = plt.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.05), frameon=False, fontsize=10
    )
    for text in leg.get_texts():
        text.set_color(COLOR_TEXTO_GRAFICO)

    output_path = os.path.join(OUTPUT_DIR, f"{sujeto_id}_01_polar_general_gold.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()


# ==========================================
# 4. FUNCIÓN 2: BARRAS DETALLADAS (Negro Oro)
# ==========================================
def crear_grafico_barras_detallado(sujeto_id, datos_preguntas):
    # (Misma lógica de preparación de datos)
    etiquetas = []
    valores = []

    for subdim_nombre, preguntas in SUBDIMENSIONES.items():
        for preg in preguntas:
            etiquetas.append(preg)
            valores.append(datos_preguntas.get(preg, 0))

    # Crear figura con fondo oscuro
    fig, ax = plt.subplots(figsize=(10, len(etiquetas) * 0.8 + 2), facecolor=COLOR_CAPA_5)
    ax.set_facecolor(COLOR_FONDO_BARRAS) # Fondo del área del gráfico

    y_positions = np.arange(len(etiquetas))

    # Barras horizontales (Oro)
    bars = ax.barh(
        y_positions,
        valores,
        align="center",
        color=COLOR_BARRA,
        height=0.6,
        zorder=2,
        edgecolor="#997A1A", # Borde oro viejo más oscuro
        linewidth=1
    )

    # Personalizar ejes (Texto Blanco)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(etiquetas, fontsize=10, fontweight="medium", color=COLOR_TEXTO_BARRAS)
    ax.invert_yaxis()

    ax.set_xlabel("Puntaje Obtenido", fontsize=10, fontweight="bold", color=COLOR_TEXTO_BARRAS)
    ax.set_xlim(0, 30)

    # Grid sutil gris
    ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=1, color="#555555")

    # Color de las líneas de los ejes (spines)
    ax.spines['bottom'].set_color('#555555')
    ax.spines['top'].set_color('#555555')
    ax.spines['right'].set_color('#555555')
    ax.spines['left'].set_color('#555555')
    ax.tick_params(axis='x', colors=COLOR_TEXTO_BARRAS) # Ticks blancos

    # Añadir el valor numérico (Texto Blanco)
    for bar in bars:
        width = bar.get_width()
        ax.text(
            width + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f"{int(width)}",
            ha="left",
            va="center",
            fontsize=9,
            fontweight="bold",
            color=COLOR_TEXTO_BARRAS,
        )

    # Título (Texto Blanco)
    plt.title(
        f"Detalle de Respuestas - {sujeto_id}",
        fontsize=13,
        fontweight="bold",
        color=COLOR_TEXTO_BARRAS,
        pad=20,
    )

    output_path = os.path.join(
        OUTPUT_DIR, f"{sujeto_id}_02_barras_detalladas_gold.png"
    )
    plt.tight_layout()
    # Guardar con el fondo oscuro de la figura
    plt.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()


# ==========================================
# 5. EJECUCIÓN PRINCIPAL CON CSV
# ==========================================
def procesar_csv_encuesta(ruta_csv):
    """Carga CSV y genera gráficos Negro/Oro para cada fila."""
    # (Se mantiene igual la lógica de importación)
    try:
        df = pd.read_csv(ruta_csv)
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo en {ruta_csv}")
        return

    # Usar la primera columna como ID
    columna_id = df.columns[0]

    print(f"Iniciando procesamiento de {len(df)} registros...")

    for index, row in df.iterrows():
        # ID del sujeto (limpiado para nombre de archivo)
        raw_id = str(row[columna_id])
        sujeto_id = "".join([c for c in raw_id if c.isalpha() or c.isdigit() or c==' ']).rstrip()
        if not sujeto_id: sujeto_id = f"Sujeto_{index+1}"

        # Extraer puntajes (CSV o Aleatorio para prueba)
        datos_preguntas = {}
        for _, preguntas in SUBDIMENSIONES.items():
            for preg in preguntas:
                if preg in row:
                    datos_preguntas[preg] = row[preg]
                else:
                    # Datos de prueba si no coincide el nombre de columna
                    datos_preguntas[preg] = np.random.randint(5, 29)

        # Calcular promedios para Radar
        puntajes_subdim = {}
        for subdim, preguntas in SUBDIMENSIONES.items():
            valores_sub = [datos_preguntas[p] for p in preguntas]
            puntajes_subdim[subdim] = np.mean(valores_sub)

        # Generar las dos gráficas Negro/Oro
        crear_grafico_polar_subdimensiones(sujeto_id, puntajes_subdim)
        crear_grafico_barras_detallado(sujeto_id, datos_preguntas)

        print(f"✓ [{index+1}/{len(df)}] Gráficas Negro/Oro generadas para: {sujeto_id}")

    print(f"\nProceso finalizado. Gráficos guardados en la carpeta: {OUTPUT_DIR}")



In [130]:
# ==========================================
# 2. MAPEO DE SUBDIMENSIONES Y PREGUNTAS
# ==========================================
# Reemplaza o ajusta los nombres de las columnas exactas de tu CSV
SUBDIMENSIONES = {
    "Dinámica Social y Estatus": ["Dinámica Social y Estatus"],
    "Ansiedad de Ejecución Institucional": ["Ansiedad de Ejecución Institucional"],
    "Identidad de Autoridad y Gestión del NO": ["Identidad de Autoridad y Gestión del NO"],
}




In [131]:
# ==========================================
# 3. FUNCIÓN 1: GRÁFICO POLAR (3 SUBDIMENSIONES)
# ==========================================
def crear_grafico_polar_subdimensiones(sujeto_id, puntajes_subdim):
    categorías = list(puntajes_subdim.keys())
    valores = list(puntajes_subdim.values())
    N = len(categorías)

    # Reordenar/cerrar el círculo
    angulos = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angulos += angulos[:1]
    valores_cerrados = valores + valores[:1]

    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 30)
    ax.grid(False)
    ax.spines["polar"].set_visible(False)

    # Capas de fondo concéntricas
    capas = [
        (30, COLOR_CAPA_5),
        (25, COLOR_CAPA_4),
        (20, COLOR_CAPA_3),
        (15, COLOR_CAPA_2),
        (10, COLOR_CAPA_1),
    ]
    for radio, color in capas:
        r = [radio] * (N + 1)
        ax.fill(angulos, r, color=color, zorder=1)

    # Polígono de datos del sujeto
    ax.plot(
        angulos,
        valores_cerrados,
        color=COLOR_PUNTAJE,
        linewidth=3.5,
        zorder=3,
        marker="o",
        markersize=7,
        label="Puntaje General",
    )

    # Etiquetas de las 3 subdimensiones
    # Ajusta texto largo con saltos de línea automáticos
    labels_format = [cat.replace(": ", ":\n") for cat in categorías]
    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(
        labels_format, fontsize=10, color="#222222", fontweight="bold"
    )

    ax.set_yticks([0, 5, 10, 15, 20, 25, 30])
    ax.set_yticklabels(
        ["0", "5", "10", "15", "20", "25", "30"], color="#333333", fontsize=8
    )
    ax.set_rlabel_position(0)

    plt.title(
        f"Perfil General por Subdimensiones - {sujeto_id}",
        fontsize=13,
        fontweight="bold",
        pad=25,
        color="#0E2A3A",
    )
    plt.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.05), frameon=False
    )

    output_path = os.path.join(OUTPUT_DIR, f"{sujeto_id}_01_polar_general.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


In [132]:


# ==========================================
# 5. EJECUCIÓN PRINCIPAL CON CSV
# ==========================================
def procesar_csv_encuesta(ruta_csv):
    # Cargar CSV y reemplazar celdas vacías por 0
    df = pd.read_csv(ruta_csv).fillna(0)

    columna_nombre = "Nombre"

    for index, row in df.iterrows():
        # Extraer el nombre de la persona (limpiando espacios)
        if columna_nombre in row and pd.notna(row[columna_nombre]):
            sujeto_id = str(row[columna_nombre]).strip()
        else:
            # Respaldo por si una fila no tiene nombre cargado
            sujeto_id = f"Sujeto_{index+1}"

        # Extraer puntajes individuales
        datos_preguntas = {}
        for subdim, preguntas in SUBDIMENSIONES.items():
            for preg in preguntas:
                datos_preguntas[preg] = (
                    row[preg] if preg in row else np.random.randint(5, 28)
                )

        # Calcular promedios/totales para la Gráfica 1 (Polar)
        puntajes_subdim = {}
        for subdim, preguntas in SUBDIMENSIONES.items():
            valores_sub = [datos_preguntas[p] for p in preguntas]
            puntajes_subdim[subdim] = np.mean(valores_sub)

        # Generar las dos gráficas con el nombre de la persona
        crear_grafico_polar_subdimensiones(sujeto_id, puntajes_subdim)

        print(f"✓ Gráficas generadas exitosamente para: {sujeto_id}")



In [133]:
procesar_csv_encuesta("T1_ Test de Rathus (respuestas) - Respuestas de formulario 1.csv")

✓ Gráficas generadas exitosamente para: Carlos Barrera
✓ Gráficas generadas exitosamente para: Jorge Mario Ramírez
✓ Gráficas generadas exitosamente para: Andrea Trujillo
✓ Gráficas generadas exitosamente para: Hugo Armando Ramírez Calvo
✓ Gráficas generadas exitosamente para: 0
✓ Gráficas generadas exitosamente para: Gisela jaramillo
✓ Gráficas generadas exitosamente para: Jessica Agudelo
✓ Gráficas generadas exitosamente para: Diego Sánchez


In [117]:
import shutil

# Borra la carpeta y todo su contenido
shutil.rmtree("graficos", ignore_errors=True)

print("¡Todos los gráficos fueron eliminados exitosamente!")

¡Todos los gráficos fueron eliminados exitosamente!
